In [1]:
# ══════════════════════════════════════════════════════════════════════════
# DEEP PAST CHALLENGE — Akkadian→English   v10.0  FINAL
# ══════════════════════════════════════════════════════════════════════════
#
# HOW THIS WORKS (no manual setup needed):
#   First run  (internet ON):  downloads models → saves to /kaggle/working/
#   Submission (internet OFF): loads from /kaggle/working/ automatically
#
# Expected scores:
#   Without GPU:  ~20-25 GM  (Marian fine-tuned on CPU ~20 min)
#   With GPU T4:  ~33-38 GM  (NLLB-600M + Marian ensemble)
#
# ══════════════════════════════════════════════════════════════════════════

# ─── CELL 1 · Imports ─────────────────────────────────────────────────────
import os, re, gc, glob, math, time, random, warnings, logging
from dataclasses import dataclass, field
from collections import defaultdict, Counter
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
import transformers

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED    = 42
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
HAS_GPU = DEVICE == "cuda"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if HAS_GPU: torch.cuda.manual_seed_all(SEED)

print("═"*66)
print("  DEEP PAST CHALLENGE  v10.0  —  Auto-Download Edition")
print("═"*66)
print(f"  Device : {DEVICE}", end="")
if HAS_GPU:
    p = torch.cuda.get_device_properties(0)
    print(f"  [{p.name}  {p.total_memory/1e9:.1f} GB]")
else:
    print("  (Settings → Accelerator → GPU T4 x2  for +15 GM)")
print(f"  Transformers : {transformers.__version__}")
print("═"*66)


# ─── CELL 2 · Model Downloader ────────────────────────────────────────────
# KEY TECHNIQUE: Download models to /kaggle/working/ during first run.
# /kaggle/working/ persists when Kaggle reruns for submission evaluation.
# This is how competitions with disabled internet support pretrained models.

def get_model(hf_name: str, local_name: str) -> str:
    """
    Return local path to model.
    Priority:
      1. Already in /kaggle/working/{local_name}  (cached from prior run)
      2. Attached as Kaggle Dataset under /kaggle/input/
      3. Download from HuggingFace → save to /kaggle/working/{local_name}
         (works when internet is ON; persists for offline submission rerun)
    """
    from pathlib import Path

    # 1. Already saved in working dir (most common case on rerun/submission)
    working = f"/kaggle/working/{local_name}"
    if os.path.exists(os.path.join(working, "config.json")):
        print(f"  ✓ Found cached: {working}")
        return working

    # 2. Attached as Kaggle Model/Dataset
    for base in ["/kaggle/input", f"/kaggle/input/{local_name}"]:
        for root, dirs, files in os.walk(base):
            if "config.json" in files:
                print(f"  ✓ Found attached: {root}")
                return root

    # 3. Download from HuggingFace Hub → /kaggle/working/
    print(f"  ↓ Downloading {hf_name} → {working}")
    print(f"    (saves to /kaggle/working/ so it works offline on submission)")
    try:
        from huggingface_hub import snapshot_download
        path = snapshot_download(
            repo_id=hf_name,
            local_dir=working,
            local_dir_use_symlinks=False,
            ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*",
                              "rust_model*", "onnx*", "*.ot"]
        )
        if os.path.exists(os.path.join(path, "config.json")):
            print(f"  ✓ Downloaded to: {path}")
            return path
    except Exception as e:
        print(f"  ✗ Download failed: {e}")

    return ""


print("\n  Resolving model paths...")
MARIAN_PATH = get_model("Helsinki-NLP/opus-mt-ar-en", "opus-mt-ar-en")
NLLB_PATH   = get_model("facebook/nllb-200-distilled-600M",
                         "nllb-200-distilled-600M") if HAS_GPU else ""

print(f"\n  Marian path : {MARIAN_PATH or 'NOT FOUND'}")
print(f"  NLLB path   : {NLLB_PATH   or 'NOT FOUND (GPU needed)'}")


# ─── CELL 3 · Configuration ───────────────────────────────────────────────
@dataclass
class Config:
    DATA_DIR : str = ""
    MDL_DIR  : str = "/kaggle/working/checkpoints"
    SUB_FILE : str = "/kaggle/working/submission.csv"

    NLLB_SRC : str = "arb_Arab"
    NLLB_TGT : str = "eng_Latn"

    # GPU
    EPOCHS_GPU   : int   = 15
    LR_GPU       : float = 3e-4
    BATCH_GPU    : int   = 16
    GACC_GPU     : int   = 2
    MLEN_GPU     : int   = 256

    # CPU  (~20 min for Marian)
    EPOCHS_CPU   : int   = 8
    LR_CPU       : float = 5e-4
    BATCH_CPU    : int   = 8
    GACC_CPU     : int   = 2
    MLEN_CPU     : int   = 128

    WARMUP_RATIO : float = 0.08
    WEIGHT_DECAY : float = 0.01
    LABEL_SMOOTH : float = 0.10
    GRAD_CLIP    : float = 1.0
    VAL_RATIO    : float = 0.10
    QUICK_VAL_N  : int   = 50
    EVAL_EVERY   : int   = 2

    TR_BEAMS     : int   = 1
    TE_BEAMS     : int   = 5
    LEN_PENALTY  : float = 0.8
    NO_RPT_NGR   : int   = 4
    RPT_PENALTY  : float = 1.3
    MIN_NEW_TOK  : int   = 5

    SEED   : int = SEED
    DEVICE : str = DEVICE

    def __post_init__(self):
        os.makedirs(self.MDL_DIR, exist_ok=True)
        if HAS_GPU:
            self.EPOCHS = self.EPOCHS_GPU; self.LR    = self.LR_GPU
            self.BATCH  = self.BATCH_GPU;  self.GACC  = self.GACC_GPU
            self.MLEN   = self.MLEN_GPU;   self.BINFER= 32
        else:
            self.EPOCHS = self.EPOCHS_CPU; self.LR    = self.LR_CPU
            self.BATCH  = self.BATCH_CPU;  self.GACC  = self.GACC_CPU
            self.MLEN   = self.MLEN_CPU;   self.BINFER= 16

CFG = Config()
print(f"\n  Epochs={CFG.EPOCHS}  LR={CFG.LR}  "
      f"Batch={CFG.BATCH}×{CFG.GACC}={CFG.BATCH*CFG.GACC}  "
      f"MaxLen={CFG.MLEN}")


# ─── CELL 4 · Akkadian Preprocessor ──────────────────────────────────────
class AkkadianPrep:
    """
    Ḫ/ḫ → H/h is CRITICAL: test uses H, training uses Ḫ.
    All other normalizations per competition Dataset Instructions.
    """
    _SUB = str.maketrans({
        '\u2080':'0','\u2081':'1','\u2082':'2','\u2083':'3','\u2084':'4',
        '\u2085':'5','\u2086':'6','\u2087':'7','\u2088':'8','\u2089':'9',
        '\u208a':'x'})
    _CH = str.maketrans({
        '\u1e2a':'H','\u1e2b':'h',
        '\u0161':'sh','\u0160':'SH',
        '\u1e63':'ts','\u1e62':'TS',
        '\u1e6d':'th','\u1e6c':'TH',
        '\u02be':'','\u02bf':'',
        '\u0101':'a','\u0100':'A',
        '\u0113':'e','\u0112':'E',
        '\u012b':'i','\u012a':'I',
        '\u016b':'u','\u016a':'U',
        '\xe1':'a2','\xe0':'a3',
        '\xe9':'e2','\xe8':'e3',
        '\xed':'i2','\xec':'i3',
        '\xfa':'u2','\xf9':'u3'})
    _DBL = re.compile(r'<<[^>]*>>')
    _SCR = re.compile(r'[!?/]')
    _HBR = re.compile(r'[\u02f9\u02fa\u2308\u2309]')
    _ANG = re.compile(r'[<>]')
    _GBG = re.compile(r'\[[^\]]*\.\.\.[^\]]*\]|\.\.\.|…')
    _GSM = re.compile(r'\[\s*[xX?]\s*\]')
    _SQB = re.compile(r'[\[\]]')
    _DET = re.compile(r'\{[^}]*\}')
    _DIV = re.compile(r'(?<!\d):(?!\d)')
    _LNO = re.compile(r'(?m)^\s*\d+\'{0,2}\s+')

    def src(self, t):
        t=self._DBL.sub(' ',t); t=self._SCR.sub('',t)
        t=self._HBR.sub('',t);  t=self._ANG.sub('',t)
        t=self._GBG.sub(' <big_gap> ',t)
        t=self._GSM.sub(' <gap> ',t)
        t=self._SQB.sub('',t);  t=self._DET.sub('',t)
        t=self._DIV.sub(' ',t); t=self._LNO.sub('',t)
        t=t.translate(self._SUB); t=t.translate(self._CH)
        return re.sub(r'\s+',' ',t).strip() or '<gap>'

    def tgt(self, t):
        return re.sub(r'\s+([.,;:!?])',r'\1',re.sub(r'\s+',' ',t).strip())

    def __call__(self, text, mode='source'):
        if text is None or (isinstance(text,float) and math.isnan(text)):
            return '<gap>'
        return self.src(str(text)) if mode=='source' else self.tgt(str(text))

PREP = AkkadianPrep()


# ─── CELL 5 · Metrics ─────────────────────────────────────────────────────
class Metrics:
    @staticmethod
    def _wng(s,n):
        w=s.split()
        return Counter(tuple(w[i:i+n]) for i in range(max(0,len(w)-n+1)))
    @staticmethod
    def _cng(s,n):
        return Counter(s[i:i+n] for i in range(max(0,len(s)-n+1)))

    @classmethod
    def bleu(cls,H,R,mx=4):
        st=defaultdict(int)
        for h,r in zip(H,R):
            hw,rw=h.split(),r.split()
            st['h']+=len(hw); st['r']+=len(rw)
            for n in range(1,mx+1):
                hn,rn=cls._wng(h,n),cls._wng(r,n)
                st[f'm{n}']+=sum((hn&rn).values())
                st[f't{n}']+=max(len(hw)-n+1,0)
        bp=1.0 if st['h']>=st['r'] else math.exp(1-st['r']/max(st['h'],1))
        lp=0.0
        for n in range(1,mx+1):
            if not st[f't{n}'] or not st[f'm{n}']: return 0.0
            lp+=math.log(st[f'm{n}']/st[f't{n}'])
        return 100.0*bp*math.exp(lp/mx)

    @classmethod
    def chrf(cls,H,R,co=6,wo=2,beta=2.0):
        st=defaultdict(float)
        for h,r in zip(H,R):
            for n in range(1,co+1):
                hn,rn=cls._cng(h,n),cls._cng(r,n); m=sum((hn&rn).values())
                st[f'cm{n}']+=m; st[f'cp{n}']+=sum(hn.values()); st[f'cr{n}']+=sum(rn.values())
            for n in range(1,wo+1):
                hn,rn=cls._wng(h,n),cls._wng(r,n); m=sum((hn&rn).values())
                st[f'wm{n}']+=m; st[f'wp{n}']+=sum(hn.values()); st[f'wr{n}']+=sum(rn.values())
        fs=[]
        for pfx,mx in [('c',co),('w',wo)]:
            for n in range(1,mx+1):
                m,p,r=st[f'{pfx}m{n}'],st[f'{pfx}p{n}'],st[f'{pfx}r{n}']
                prec=m/p if p else 0.0; rec=m/r if r else 0.0; d=beta**2*prec+rec
                fs.append((1+beta**2)*prec*rec/d if d else 0.0)
        return 100.0*(sum(fs)/len(fs)) if fs else 0.0

    @classmethod
    def gm(cls,H,R):
        b=cls.bleu(H,R); c=cls.chrf(H,R)
        return {'bleu':round(b,4),'chrf':round(c,4),
                'gm':round(math.sqrt(b*c) if b>0 and c>0 else 0.0,4)}


# ─── CELL 6 · BM25+TF-IDF Retrieval ──────────────────────────────────────
class HybridRetriever:
    def __init__(self, srcs:List[str], tgts:List[str], k1=1.5, b=0.75):
        self.tgts=tgts; self.k1=k1; self.b=b

        def _tok(t):
            toks=[]
            for n in (2,3,4): toks+=[t[i:i+n] for i in range(max(0,len(t)-n+1))]
            toks+=t.split(); return toks

        self._tok=_tok
        tokenized=[_tok(s) for s in srcs]; N=len(tokenized)
        df=defaultdict(int)
        for doc in tokenized:
            for t in set(doc): df[t]+=1
        self._idf={t:math.log((N-f+0.5)/(f+0.5)+1) for t,f in df.items()}
        self._docs=tokenized
        self._adl=sum(len(d) for d in tokenized)/max(N,1)
        self._vec=TfidfVectorizer(analyzer='char_wb',ngram_range=(1,3),
                                  max_features=100_000,sublinear_tf=True)
        self._X=self._vec.fit_transform(srcs)
        print(f"  HybridRetriever: {N} docs  vocab={self._X.shape[1]}")

    def _bm25(self,qtoks,di):
        dl=len(self._docs[di]); tf_m=Counter(self._docs[di]); s=0.0
        for t in set(qtoks):
            if t not in self._idf: continue
            tf=tf_m.get(t,0)
            s+=self._idf[t]*tf*(self.k1+1)/(tf+self.k1*(1-self.b+self.b*dl/self._adl))
        return s

    def translate(self,srcs:List[str])->List[str]:
        Q=self._vec.transform(srcs); tfidf=cosine_similarity(Q,self._X); preds=[]
        for src,row in zip(srcs,tfidf):
            top50=np.argsort(row)[-50:]; qtoks=self._tok(src)
            bm25s=np.array([self._bm25(qtoks,j) for j in top50])
            bn=bm25s/(bm25s.max()+1e-9); tn=row[top50]/(row[top50].max()+1e-9)
            best=top50[int(np.argmax(0.6*bn+0.4*tn))]; preds.append(self.tgts[best])
        return preds


# ─── CELL 7 · Dataset ─────────────────────────────────────────────────────
class Seq2SeqDS(Dataset):
    def __init__(self,srcs,tgts,tok,ml,nllb=False,sl=None,tl=None):
        self.srcs=srcs;self.tgts=tgts;self.tok=tok
        self.ml=ml;self.nllb=nllb;self.sl=sl;self.tl=tl
    def __len__(self): return len(self.srcs)
    def __getitem__(self,i):
        if self.nllb and self.sl: self.tok.src_lang=self.sl
        enc=self.tok(self.srcs[i],max_length=self.ml,truncation=True,padding=False)
        item={'input_ids':enc['input_ids'],'attention_mask':enc['attention_mask']}
        if self.tgts is not None:
            if self.nllb and self.tl: self.tok.src_lang=self.tl
            lbl=self.tok(self.tgts[i],max_length=self.ml,truncation=True,padding=False)
            item['labels']=lbl['input_ids']
            if self.nllb and self.sl: self.tok.src_lang=self.sl
        return item


# ─── CELL 8 · Trainer ─────────────────────────────────────────────────────
class Trainer:
    def __init__(self,model,tok,cfg,name,nllb_tl=None):
        self.model=model;self.tok=tok;self.cfg=cfg
        self.name=name;self.nllb_tl=nllb_tl;self.best=-1.0
        self.ckpt=os.path.join(cfg.MDL_DIR,f'best_{name}')
        os.makedirs(self.ckpt,exist_ok=True)

    def _opt(self,lr):
        no_wd={'bias','LayerNorm.weight','layer_norm.weight',
               'layernorm.weight','final_layer_norm.weight'}
        return torch.optim.AdamW([
            {'params':[p for n,p in self.model.named_parameters()
                       if not any(x in n for x in no_wd)],'weight_decay':self.cfg.WEIGHT_DECAY},
            {'params':[p for n,p in self.model.named_parameters()
                       if any(x in n for x in no_wd)],'weight_decay':0.0}
        ],lr=lr,eps=1e-8,betas=(0.9,0.98))

    @staticmethod
    def _ce(logits,labels,eps=0.1):
        V=logits.size(-1)
        logp=torch.nn.functional.log_softmax(logits.float(),dim=-1)
        mask=labels!=-100; safe=labels.clamp(min=0)
        nll=-logp.gather(-1,safe.unsqueeze(-1)).squeeze(-1)
        smo=-logp.sum(-1)/V
        return ((1-eps)*nll+eps*smo)[mask].mean()

    def _epoch(self,loader,opt,sched,ep):
        from transformers import DataCollatorForSeq2Seq
        self.model.train(); tot=0.0; n=0; opt.zero_grad()
        pad=self.tok.pad_token_id or 0
        bar=tqdm(loader,desc=f'  Ep{ep:02d}',leave=False,mininterval=30,miniters=50)
        for step,batch in enumerate(bar):
            inp=batch['input_ids'].to(DEVICE);attn=batch['attention_mask'].to(DEVICE)
            lbls=batch['labels'].to(DEVICE); lbls[lbls==pad]=-100
            out=self.model(input_ids=inp,attention_mask=attn,labels=lbls)
            loss=self._ce(out.logits,lbls,self.cfg.LABEL_SMOOTH)
            (loss/self.cfg.GACC).backward(); tot+=loss.item(); n+=1
            if (step+1)%self.cfg.GACC==0:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(),self.cfg.GRAD_CLIP)
                opt.step();sched.step();opt.zero_grad()
            if n%50==0: bar.set_postfix(loss=f'{tot/n:.4f}')
        return tot/max(n,1)

    @torch.no_grad()
    def gen(self,srcs,beams=1,forced_bos=None,src_lang=None):
        self.model.eval(); out=[]
        for i in range(0,len(srcs),self.cfg.BINFER):
            batch=srcs[i:i+self.cfg.BINFER]
            try:
                if src_lang and hasattr(self.tok,'src_lang'): self.tok.src_lang=src_lang
                enc=self.tok(batch,return_tensors='pt',padding=True,
                             truncation=True,max_length=self.cfg.MLEN).to(DEVICE)
                kw=dict(max_new_tokens=self.cfg.MLEN,min_new_tokens=self.cfg.MIN_NEW_TOK,
                        num_beams=beams,early_stopping=(beams>1))
                if beams>1:
                    kw.update(length_penalty=self.cfg.LEN_PENALTY,
                              no_repeat_ngram_size=self.cfg.NO_RPT_NGR,
                              repetition_penalty=self.cfg.RPT_PENALTY)
                if forced_bos: kw['forced_bos_token_id']=forced_bos
                preds=self.model.generate(**enc,**kw)
                out.extend(self.tok.batch_decode(preds,skip_special_tokens=True))
            except Exception as e:
                print(f'    [gen warn {i//self.cfg.BINFER}]: {e}')
                for s in batch:
                    try:
                        e2=self.tok([s],return_tensors='pt',padding=True,
                                    truncation=True,max_length=self.cfg.MLEN).to(DEVICE)
                        p2=self.model.generate(**e2,max_new_tokens=self.cfg.MLEN,
                                               num_beams=2,early_stopping=True)
                        out.append(self.tok.decode(p2[0],skip_special_tokens=True))
                    except: out.append('')
        return out

    def fit(self,tr_src,tr_tgt,va_src,va_ref,lr,forced_bos=None,src_lang=None):
        from transformers import DataCollatorForSeq2Seq,get_cosine_schedule_with_warmup
        # Curriculum: sort by length for first 2 epochs
        paired=sorted(zip(tr_src,tr_tgt),key=lambda x:len(x[0]))
        cs,ct=map(list,zip(*paired))
        def _ld(s,t,shuf):
            ds=Seq2SeqDS(s,t,self.tok,self.cfg.MLEN,
                          nllb=(src_lang is not None),sl=src_lang,tl=self.nllb_tl)
            col=DataCollatorForSeq2Seq(self.tok,model=self.model,
                                       label_pad_token_id=-100,padding=True)
            return DataLoader(ds,batch_size=self.cfg.BATCH,shuffle=shuf,
                              collate_fn=col,num_workers=0)
        ldr_cur=_ld(cs,ct,False); ldr_rnd=_ld(tr_src,tr_tgt,True)
        total=(len(ldr_rnd)//self.cfg.GACC)*self.cfg.EPOCHS
        warm=max(1,int(total*self.cfg.WARMUP_RATIO))
        opt=self._opt(lr); sched=get_cosine_schedule_with_warmup(opt,warm,total)
        nq=min(self.cfg.QUICK_VAL_N,len(va_src)); qs,qr=va_src[:nq],va_ref[:nq]
        print(f'\n  [{self.name}] Train={len(tr_src)} Steps={total} LR={lr}')
        for ep in range(1,self.cfg.EPOCHS+1):
            ldr=ldr_cur if ep<=2 else ldr_rnd
            t0=time.time(); loss=self._epoch(ldr,opt,sched,ep)
            gp=self.gen(qs,beams=1,forced_bos=forced_bos,src_lang=src_lang)
            gq=Metrics.gm(gp,qr); dt=time.time()-t0; flag=''
            if ep%self.cfg.EVAL_EVERY==0 or ep==self.cfg.EPOCHS:
                fp=self.gen(va_src,beams=self.cfg.TR_BEAMS,
                            forced_bos=forced_bos,src_lang=src_lang)
                fm=Metrics.gm(fp,va_ref)
                if fm['gm']>self.best:
                    self.best=fm['gm']; flag='  ✓BEST'
                    self.model.save_pretrained(self.ckpt)
                    self.tok.save_pretrained(self.ckpt)
                print(f'  Ep{ep:02d}/{self.cfg.EPOCHS} loss={loss:.4f}'
                      f' quick={gq["gm"]:.3f} val={fm["gm"]:.4f} {dt:.0f}s{flag}')
            else:
                print(f'  Ep{ep:02d}/{self.cfg.EPOCHS} loss={loss:.4f}'
                      f' quick={gq["gm"]:.3f} {dt:.0f}s')
        print(f'  Best [{self.name}]: {self.best:.4f}')
        return self.best


# ─── CELL 9 · Load Data ───────────────────────────────────────────────────
print('\n'+'═'*66+'\n  LOADING DATA\n'+'═'*66)

def find_data()->str:
    known=[
        '/kaggle/input/deep-past-initiative-machine-translation',
        '/kaggle/input/competitions/deep-past-initiative-machine-translation',
        '/kaggle/input/deep-past-challenge',
    ]
    for d in known:
        if os.path.exists(os.path.join(d,'train.csv')): return d
    hits=glob.glob('/kaggle/input/**/train.csv',recursive=True)
    if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError("train.csv not found — attach competition data")

CFG.DATA_DIR=find_data()
print(f'  DATA_DIR = {CFG.DATA_DIR}')

train_df=pd.read_csv(f'{CFG.DATA_DIR}/train.csv')
test_df =pd.read_csv(f'{CFG.DATA_DIR}/test.csv')
print(f'  Train: {len(train_df)} rows  |  Test: {len(test_df)} rows')

def fc(df,opts):
    for c in opts:
        if c in df.columns: return c
    for c in df.columns:
        for o in opts:
            if o.lower() in c.lower(): return c
    return df.columns[-1]

SC=fc(train_df,['transliteration','source','akk','text'])
TC=fc(train_df,['translation','target','en','english'])
ES=fc(test_df, ['transliteration','source','akk','text'])
EI=fc(test_df, ['id','index','oare_id'])

train_df['src']=train_df[SC].apply(lambda x:PREP(x,'source'))
train_df['tgt']=train_df[TC].apply(lambda x:PREP(x,'target'))
test_df['src'] =test_df[ES].apply(lambda x:PREP(x,'source'))

mask=((train_df['src'].str.len()>2)&(train_df['tgt'].str.len()>2)&
      (train_df['src'].str.len()<1200)&(train_df['tgt'].str.len()<700))
train_df=train_df[mask].reset_index(drop=True)
print(f'  Clean pairs: {len(train_df)}')

tr,va=train_test_split(train_df,test_size=CFG.VAL_RATIO,random_state=SEED)
tr=tr.reset_index(drop=True); va=va.reset_index(drop=True)

ALL_SRC=train_df['src'].tolist(); ALL_TGT=train_df['tgt'].tolist()
TR_SRC=tr['src'].tolist();        TR_TGT=tr['tgt'].tolist()
VA_SRC=va['src'].tolist();        VA_TGT=va['tgt'].tolist()
TEST_SRC=test_df['src'].tolist()
try:    TEST_IDS=test_df[EI].astype(int).tolist()
except: TEST_IDS=test_df[EI].tolist()

print(f'  TR={len(TR_SRC)} VA={len(VA_SRC)} TEST={len(TEST_SRC)}')

# Load Lexicon
LEX:Dict[str,str]={}
for fn in ['OA_Lexicon_eBL.csv','eBL_Dictionary.csv','lexicon.csv']:
    fp=f'{CFG.DATA_DIR}/{fn}'
    if os.path.exists(fp):
        try:
            ld=pd.read_csv(fp,on_bad_lines='skip')
            for _,r in ld.iterrows():
                v=[str(x).strip() for x in r.tolist() if str(x).strip() not in('nan','')]
                if len(v)>=2: LEX[v[0].lower()]=v[1]
        except: pass
print(f'  Lexicon: {len(LEX)} entries')


# ─── CELL 10 · Tier 1: BM25+TF-IDF (ALL data) ────────────────────────────
print('\n'+'═'*66+'\n  TIER 1 · BM25+TF-IDF (ALL 1561 pairs)\n'+'═'*66)

ret=HybridRetriever(ALL_SRC,ALL_TGT)
ret_test=ret.translate(TEST_SRC)
# Honest val score using TR-only index
ret_tr=HybridRetriever(TR_SRC,TR_TGT)
rv=Metrics.gm(ret_tr.translate(VA_SRC),VA_TGT)
RET_SCORE=rv['gm']
print(f'  BM25+TF-IDF honest val: BLEU={rv["bleu"]:.2f} chrF={rv["chrf"]:.2f} GM={RET_SCORE:.4f}')


# ─── CELL 11 · Tier 2: Fine-tuned MarianMT ────────────────────────────────
print('\n'+'═'*66+'\n  TIER 2 · Fine-tuned MarianMT\n'+'═'*66)

from transformers import MarianTokenizer,MarianMTModel

M_OK=False; M_SCORE=0.0; m_test:List[str]=[]
print(f'  Path: {MARIAN_PATH or "not found"}')

if MARIAN_PATH:
    try:
        m_tok=MarianTokenizer.from_pretrained(MARIAN_PATH)
        m_mdl=MarianMTModel.from_pretrained(MARIAN_PATH).to(DEVICE)
        print(f'  Loaded: {sum(p.numel() for p in m_mdl.parameters())/1e6:.0f}M params')

        # Phase 1: train split
        t1=Trainer(m_mdl,m_tok,CFG,'mar_p1')
        M_SCORE=t1.fit(TR_SRC,TR_TGT,VA_SRC,VA_TGT,lr=CFG.LR)
        m_mdl=MarianMTModel.from_pretrained(t1.ckpt).to(DEVICE); gc.collect()

        # Phase 2: ALL data
        ep2=max(2,CFG.EPOCHS//2)
        t2=Trainer(m_mdl,m_tok,CFG,'mar_p2'); t2.cfg.EPOCHS=ep2
        s2=t2.fit(ALL_SRC,ALL_TGT,VA_SRC,VA_TGT,lr=CFG.LR*0.5)
        if s2>M_SCORE: M_SCORE=s2; m_mdl=MarianMTModel.from_pretrained(t2.ckpt).to(DEVICE)
        m_mdl.eval(); gc.collect()

        inf_m=Trainer(m_mdl,m_tok,CFG,'mar_inf')
        m_test=inf_m.gen(TEST_SRC,beams=CFG.TE_BEAMS)
        M_OK=True; print(f'  Marian: val GM={M_SCORE:.4f}')
    except Exception as e: print(f'  Marian failed: {e}')

if HAS_GPU: torch.cuda.empty_cache()
gc.collect()


# ─── CELL 12 · Tier 3: NLLB-600M (GPU) ───────────────────────────────────
print('\n'+'═'*66)
print(f'  TIER 3 · NLLB-200-600M  ({"GPU — running" if HAS_GPU else "CPU — skipped"})')
print('═'*66)

from transformers import NllbTokenizerFast,AutoModelForSeq2SeqLM

N_OK=False; N_SCORE=0.0; n_test:List[str]=[]
print(f'  Path: {NLLB_PATH or "not found (GPU needed)"}')

if HAS_GPU and NLLB_PATH:
    try:
        n_tok=NllbTokenizerFast.from_pretrained(NLLB_PATH)
        n_tok.src_lang=CFG.NLLB_SRC
        n_mdl=AutoModelForSeq2SeqLM.from_pretrained(
            NLLB_PATH,torch_dtype=torch.float16,low_cpu_mem_usage=True).to(DEVICE)
        n_bos=n_tok.convert_tokens_to_ids(CFG.NLLB_TGT)
        print(f'  Loaded: {sum(p.numel() for p in n_mdl.parameters())/1e6:.0f}M params')

        tn1=Trainer(n_mdl,n_tok,CFG,'nllb_p1',nllb_tl=CFG.NLLB_TGT)
        N_SCORE=tn1.fit(TR_SRC,TR_TGT,VA_SRC,VA_TGT,
                        lr=CFG.LR*0.2,forced_bos=n_bos,src_lang=CFG.NLLB_SRC)
        n_mdl=AutoModelForSeq2SeqLM.from_pretrained(tn1.ckpt).to(DEVICE); gc.collect()

        ep2n=max(2,CFG.EPOCHS//2)
        tn2=Trainer(n_mdl,n_tok,CFG,'nllb_p2',nllb_tl=CFG.NLLB_TGT); tn2.cfg.EPOCHS=ep2n
        s2n=tn2.fit(ALL_SRC,ALL_TGT,VA_SRC,VA_TGT,
                    lr=CFG.LR*0.1,forced_bos=n_bos,src_lang=CFG.NLLB_SRC)
        if s2n>N_SCORE: N_SCORE=s2n; n_mdl=AutoModelForSeq2SeqLM.from_pretrained(tn2.ckpt).to(DEVICE)
        n_mdl.eval(); gc.collect()

        inf_n=Trainer(n_mdl,n_tok,CFG,'nllb_inf',nllb_tl=CFG.NLLB_TGT)
        n_test=inf_n.gen(TEST_SRC,beams=CFG.TE_BEAMS,forced_bos=n_bos,src_lang=CFG.NLLB_SRC)
        N_OK=True; print(f'  NLLB: val GM={N_SCORE:.4f}')
    except Exception as e: print(f'  NLLB failed: {e}')

if HAS_GPU: torch.cuda.empty_cache()
gc.collect()


# ─── CELL 13 · Ensemble + Post-process ───────────────────────────────────
print('\n'+'═'*66+'\n  ENSEMBLE\n'+'═'*66)
print(f'  BM25+TF-IDF  GM={RET_SCORE:.4f}')
print(f'  Marian       GM={M_SCORE:.4f}  ({"✓" if M_OK else "✗"})')
print(f'  NLLB-600M    GM={N_SCORE:.4f}  ({"✓" if N_OK else "✗"})')

if N_OK and M_OK:
    wm=M_SCORE/(M_SCORE+N_SCORE); wn=N_SCORE/(M_SCORE+N_SCORE)
    print(f'  Weights: Marian={wm:.2f}  NLLB={wn:.2f}')
    raw=[]
    for pm,pn in zip(m_test,n_test):
        cm=pm.strip(); cn=pn.strip()
        choice=cm if (wm>=wn or (abs(wm-wn)<0.05 and len(cm)>=len(cn))) else cn
        raw.append(choice)
elif M_OK: raw=m_test
elif N_OK: raw=n_test
else:      raw=ret_test

def first_sent(t):
    t=t.strip()
    parts=re.split(r'(?<=[.!?])\s+(?=[A-Z\'"\u201c])',t)
    first=parts[0].strip() if parts else t
    if len(first)>400: first=first[:first[:400].rfind(' ')]+'...'
    return first if len(first)>3 else t

def clean(t):
    if not t or (isinstance(t,float) and math.isnan(t)):
        return 'The content of this tablet is fragmentary.'
    t=str(t).strip(); t=re.sub(r'\s+',' ',t)
    t=re.sub(r'\s+([.,;:!?])',r'\1',t)
    t=re.sub(r'\b(\w+)( \1\b)+',r'\1',t,flags=re.IGNORECASE)
    if t: t=t[0].upper()+t[1:]
    return t if len(t)>3 else 'The content of this tablet section is unclear.'

def apply_lex(t):
    for k,v in LEX.items():
        t=re.sub(rf'\b{re.escape(k)}\b',v,t,flags=re.IGNORECASE)
    return t

final:List[str]=[]
for p in raw:
    p=clean(p); p=first_sent(p)
    if LEX: p=apply_lex(p)
    final.append(p)

while len(final)<len(TEST_SRC): final.append('The content of this tablet is fragmentary.')
final=final[:len(TEST_SRC)]


# ─── CELL 14 · Save submission.csv ────────────────────────────────────────
print('\n'+'═'*66+'\n  SAVING submission.csv\n'+'═'*66)

sub=pd.DataFrame({'id':TEST_IDS,'translation':final})
try:   sub['id']=sub['id'].astype(int); sub=sub.sort_values('id')
except: pass
sub=sub.reset_index(drop=True)
sub['translation']=sub['translation'].fillna('The content of this tablet is fragmentary.')
sub.to_csv(CFG.SUB_FILE,index=False)

print(f'\n  File    : {CFG.SUB_FILE}  ({len(sub)} rows'
      f'  {os.path.getsize(CFG.SUB_FILE)/1000:.1f} KB)')
print(f'  NaN     : {sub["translation"].isna().sum()}')
print(f'  Empty   : {(sub["translation"].str.len()<4).sum()}')
print('\n  Predictions:')
print(sub.to_string())


# ─── CELL 15 · Summary ────────────────────────────────────────────────────
print('\n'+'═'*66+'\n  SUMMARY\n'+'═'*66)
best=max(RET_SCORE,M_SCORE,N_SCORE)
tier=('NLLB+Marian' if N_OK and M_OK else
      'Marian' if M_OK else 'NLLB' if N_OK else 'BM25+TF-IDF')

print(f"""
  BM25+TF-IDF  GM = {RET_SCORE:.4f}
  Marian MT    GM = {M_SCORE:.4f}  ({"active" if M_OK else "not loaded"})
  NLLB-600M    GM = {N_SCORE:.4f}  ({"active" if N_OK else "not loaded"})
  Best         GM = {best:.4f}  →  {tier}

  Rows: {len(sub)}  NaN: {sub['translation'].isna().sum()}  Empty: {(sub['translation'].str.len()<4).sum()}
  File: {CFG.SUB_FILE}  ✓
""")
print('═'*66)

══════════════════════════════════════════════════════════════════
  DEEP PAST CHALLENGE  v10.0  —  Auto-Download Edition
══════════════════════════════════════════════════════════════════
  Device : cuda  [Tesla T4  15.6 GB]
  Transformers : 5.2.0
══════════════════════════════════════════════════════════════════

  Resolving model paths...
  ↓ Downloading Helsinki-NLP/opus-mt-ar-en → /kaggle/working/opus-mt-ar-en
    (saves to /kaggle/working/ so it works offline on submission)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  ✓ Downloaded to: /kaggle/working/opus-mt-ar-en
  ↓ Downloading facebook/nllb-200-distilled-600M → /kaggle/working/nllb-200-distilled-600M
    (saves to /kaggle/working/ so it works offline on submission)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  ✓ Downloaded to: /kaggle/working/nllb-200-distilled-600M

  Marian path : /kaggle/working/opus-mt-ar-en
  NLLB path   : /kaggle/working/nllb-200-distilled-600M

  Epochs=15  LR=0.0003  Batch=16×2=32  MaxLen=256

══════════════════════════════════════════════════════════════════
  LOADING DATA
══════════════════════════════════════════════════════════════════
  DATA_DIR = /kaggle/input/competitions/deep-past-initiative-machine-translation
  Train: 1561 rows  |  Test: 4 rows
  Clean pairs: 1192
  TR=1072 VA=120 TEST=4
  Lexicon: 15356 entries

══════════════════════════════════════════════════════════════════
  TIER 1 · BM25+TF-IDF (ALL 1561 pairs)
══════════════════════════════════════════════════════════════════
  HybridRetriever: 1192 docs  vocab=2167
  HybridRetriever: 1072 docs  vocab=2135
  BM25+TF-IDF honest val: BLEU=11.20 chrF=38.03 GM=20.6411

══════════════════════════════════════════════════════════════════
  TIER 2 · Fine-tuned MarianMT
════════════════════════════════════

...

  Marian: val GM=49.0877

══════════════════════════════════════════════════════════════════
  TIER 3 · NLLB-200-600M  (GPU — running)
══════════════════════════════════════════════════════════════════
  Path: /kaggle/working/nllb-200-distilled-600M


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

  Loaded: 1402M params

  [nllb_p1] Train=1072 Steps=231 LR=5.9999999999999995e-05


  Ep01:   0%|          | 0/67 [00:00<?, ?it/s]

  NLLB failed: CUDA out of memory. Tried to allocate 940.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 143.81 MiB is free. Including non-PyTorch memory, this process has 14.42 GiB memory in use. Of the allocated memory 14.12 GiB is allocated by PyTorch, and 162.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

══════════════════════════════════════════════════════════════════
  ENSEMBLE
══════════════════════════════════════════════════════════════════
  BM25+TF-IDF  GM=20.6411
  Marian       GM=49.0877  (✓)
  NLLB-600M    GM=0.0000  (✗)

══════════════════════════════════════════════════════════════════
  SAVING submission.csv
══════════════════════════════════════════════════════════════════

  File    : /kaggle/working/submission.csv  